# 02 — 设备与实例管理

本笔记本介绍如何发现、选择并连接 Ivium 设备，包括同时运行多个 IviumSoft 实例的场景。

### 概念

- **IviumSoft 实例**：一个运行中的 IviumSoft 应用副本，最多可同时运行 32 个。
- **设备 / 通道**：连接到某个实例的物理恒电位仪。
- **序列号**：印在每台 Ivium 硬件单元上的唯一标识符。

```
 IviumSoft 实例 1  ──►  CompactStat #A  (SN: 12345)
 IviumSoft 实例 2  ──►  CompactStat #B  (SN: 67890)
 IviumSoft 实例 3  ──►  （无设备）
```

Pyvium **每次只与一个实例通信**。通过 `select_iviumsoft_instance()` 切换实例。

> **错误处理：** 所有 Pyvium 异常均已在 `01_getting_started.ipynb` 第 4 节中说明。本笔记本为简洁起见，统一使用 `except Exception as e`。

In [ ]:
from pyvium import Pyvium

## 1. 发现正在运行的 IviumSoft 实例

In [ ]:
Pyvium.open_driver()

instances = Pyvium.get_active_iviumsoft_instances()
max_devices = Pyvium.get_max_device_number()

print(f"活跃 IviumSoft 实例数 : {instances}")
print(f"最大可管理设备数     : {max_devices}")

## 2. 选择实例

后续所有调用都在**当前选中的**实例上执行。如果只打开了一个 IviumSoft，此步骤为可选 — 默认选中实例 1。

In [ ]:
# 选择实例 1（第一个运行中的 IviumSoft）
Pyvium.select_iviumsoft_instance(1)

status_code, status_label = Pyvium.get_device_status()
print(f"实例 1 — 状态: {status_code} ('{status_label}')")

# 状态码含义：
#  -1 : 该槽位无 IviumSoft
#   0 : IviumSoft 运行中但设备未连接
#   1 : 设备已连接且空闲
#   2 : 设备已连接且忙碌（正在运行方法）
#   3 : USB 上未检测到设备

## 3. 遍历所有实例

并行运行多台设备时，遍历所有活跃实例以获取各自状态。

In [ ]:
instances = Pyvium.get_active_iviumsoft_instances()

for instance_number in instances:
    Pyvium.select_iviumsoft_instance(instance_number)
    code, label = Pyvium.get_device_status()
    print(f"  实例 {instance_number}: {label}")

## 4. 连接设备

「连接」操作告知 IviumSoft 接管物理设备控制权。设备必须先在 USB 上被检测到（状态 != 3）。

> **副作用：** `connect_device()` 始终会将仪器的**自动电流量程重置为关闭**，以建立已知状态。如果你依赖自动电流量程保持激活，请在连接后显式调用 `set_bistat_mode(1)` 重新启用（参见笔记本 `05_bipotentiostat_and_we32.ipynb`）。

In [ ]:
Pyvium.select_iviumsoft_instance(1)

code, label = Pyvium.get_device_status()
print(f"连接前: {label}")

if code == 0:  # IviumSoft 运行中但设备尚未连接
    Pyvium.connect_device()
    code, label = Pyvium.get_device_status()
    print(f"连接后 : {label}")
elif code == 1:
    print("设备已连接且空闲")
elif code == 2:
    print("设备已连接且忙碌")
elif code == 3:
    print("未检测到设备 — 请检查 USB 线缆")

## 5. 读取序列号

序列号可唯一标识每台物理设备。在使用多台设备时，有助于确认当前活跃的是哪台。

In [ ]:
serial = Pyvium.get_device_serial_number()
print(f"设备序列号: {serial}")

## 6. 按序列号选择设备

`select_serial_number()` 是根据**特定物理设备**进行定位的推荐方式，不依赖槽位号（槽位号可能因设备拔插而变化），更为可靠。

返回该设备在 IviumSoft 下拉列表中的从 0 开始的索引。

In [ ]:
TARGET_SERIAL = "12345"  # 替换为你设备的序列号

index = Pyvium.select_serial_number(TARGET_SERIAL)
print(f"设备 '{TARGET_SERIAL}' 位于下拉列表索引 {index}")

Pyvium.connect_device()
print("设备已连接")

## 7. 多通道：选择通道

Ivium-n-Soft（多通道版本）在一个 IviumSoft 实例中管理多个通道。`select_channel()` 打开对应标签页并激活该通道。

这与 `select_iviumsoft_instance()` **不同** — 通道选择是在单个 IviumSoft 窗口内切换，而实例选择是在不同的 IviumSoft 窗口之间切换。

In [ ]:
CHANNEL = 1  # 通道从 1 开始编号

Pyvium.select_channel(CHANNEL)
print(f"通道 {CHANNEL} 已选择")

Pyvium.connect_device()
print("该通道设备已连接")

## 8. 断开连接

调用 `disconnect_device()` 可释放 IviumSoft 对设备的控制权，而不关闭 IviumSoft。适用于多设备工作流中需要移交控制权的场景。

In [ ]:
Pyvium.disconnect_device()
print("设备已断开")

code, label = Pyvium.get_device_status()
print(f"断开后状态: {label}")

## 9. 完整多实例工作流

模板：遍历所有活跃实例，逐一连接并操作，然后继续下一个。

In [ ]:
import time

try:
    Pyvium.open_driver()
    instances = Pyvium.get_active_iviumsoft_instances()
    print(f"找到 {len(instances)} 个活跃实例: {instances}")

    for inst in instances:
        Pyvium.select_iviumsoft_instance(inst)
        code, label = Pyvium.get_device_status()
        print(f"\n实例 {inst}: {label}")

        if code == 0:
            Pyvium.connect_device()
            serial = Pyvium.get_device_serial_number()
            print(f"  已连接 — 序列号: {serial}")

            # ... 在此处对该设备执行操作 ...

            Pyvium.disconnect_device()
            print(f"  已断开")

        elif code == 1:
            serial = Pyvium.get_device_serial_number()
            print(f"  已连接 — 序列号: {serial}")

        elif code == 2:
            print(f"  忙碌中 — 跳过")

        elif code == 3:
            print(f"  USB 上无设备 — 跳过")

finally:
    Pyvium.close_driver()

---

## 小结

| 任务 | 方法 |
|------|--------|
| 列出运行中的 IviumSoft 窗口 | `get_active_iviumsoft_instances()` |
| 切换 IviumSoft 窗口 | `select_iviumsoft_instance(n)` |
| 获取设备状态 | `get_device_status()` → `(code, label)` |
| 连接所选设备 | `connect_device()` |
| 断开设备 | `disconnect_device()` |
| 获取序列号 | `get_device_serial_number()` |
| 按序列号选择设备 | `select_serial_number(sn)` |
| 选择多通道标签页 | `select_channel(n)` |

## 下一步

- **`03_direct_mode_basics.ipynb`** — 直接设置电位、读取电流、控制电池